# 02 · Labels
Map UrbanSound8K classes to CityPulse app-level labels and persist.

In [1]:
# interpreter & imports
import sys
print(sys.executable)

import pandas as pd
from pathlib import Path
print('imports OK')

c:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
imports OK


In [2]:
# load metadata
ROOT = Path.cwd().parent
meta = pd.read_csv(ROOT / 'data' / 'raw' / 'UrbanSound8K' / 'metadata' / 'UrbanSound8K.csv')
print('ROOT:', ROOT)
print('CSV shape:', meta.shape)
print('Columns:', meta.columns.tolist())

ROOT: c:\Users\LENOVO\Desktop\CityPulse
CSV shape: (8732, 8)
Columns: ['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold', 'classID', 'class']


In [3]:
# mapping and filter
CLASS_MAP = {
    'car_horn':       'traffic_horn',
    'engine_idling':  'traffic_horn',
    'drilling':       'construction',
    'jackhammer':     'construction',
    'siren':          'siren',
    'street_music':   'music_loudspeaker',
    'air_conditioner': 'hvac',
}

kept = meta[meta['class'].isin(CLASS_MAP)].copy()
dropped = meta[~meta['class'].isin(CLASS_MAP)].copy()

print(f'Rows kept   : {len(kept)}')
print(f'Rows dropped: {len(dropped)}')
print(f'Sum         : {len(kept) + len(dropped)}')

Rows kept   : 6358
Rows dropped: 2374
Sum         : 8732


In [4]:
# build labelled DataFrame
df = pd.DataFrame({
    'path': (
        'data/raw/UrbanSound8K/audio/fold'
        + kept['fold'].astype(str)
        + '/'
        + kept['slice_file_name']
    ),
    'source':         'us8k',
    'fold':           kept['fold'].values,
    'original_class': kept['class'].values,
    'app_class':      kept['class'].map(CLASS_MAP).values,
})
df = df.reset_index(drop=True)
print('DataFrame shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head(3).to_string())

DataFrame shape: (6358, 5)
Columns: ['path', 'source', 'fold', 'original_class', 'app_class']
                                                  path source  fold original_class     app_class
0  data/raw/UrbanSound8K/audio/fold10/100648-1-0-0.wav   us8k    10       car_horn  traffic_horn
1  data/raw/UrbanSound8K/audio/fold10/100648-1-1-0.wav   us8k    10       car_horn  traffic_horn
2  data/raw/UrbanSound8K/audio/fold10/100648-1-2-0.wav   us8k    10       car_horn  traffic_horn


In [5]:
# verify paths exist, no duplicates, no NaN
exists_mask  = df['path'].apply(lambda p: (ROOT / p).exists())
n_exist      = exists_mask.sum()
n_missing    = (~exists_mask).sum()
n_duplicates = df['path'].duplicated().sum()
nan_counts   = df.isna().sum()

print(f'Files that exist : {n_exist}')
print(f'Files missing    : {n_missing}')
print(f'Duplicate paths  : {n_duplicates}')
print('NaN count per column:')
print(nan_counts.to_string())

if n_missing > 0:
    print('MISSING FILES:')
    print(df.loc[~exists_mask, 'path'].to_string())

Files that exist : 6358
Files missing    : 0
Duplicate paths  : 0
NaN count per column:
path              0
source            0
fold              0
original_class    0
app_class         0


In [6]:
# distribution summary
print('=== Row count per app_class ===')
print(df['app_class'].value_counts().to_string())

print('\n=== Row count per original_class ===')
print(df['original_class'].value_counts().to_string())

print('\n=== Crosstab: app_class vs fold ===')
ct = pd.crosstab(df['app_class'], df['fold'])
print(ct.to_string())

=== Row count per app_class ===
app_class
construction         2000
traffic_horn         1429
hvac                 1000
music_loudspeaker    1000
siren                 929

=== Row count per original_class ===
original_class
air_conditioner    1000
street_music       1000
engine_idling      1000
jackhammer         1000
drilling           1000
siren               929
car_horn            429

=== Crosstab: app_class vs fold ===
fold                1    2    3    4    5    6    7    8    9    10
app_class                                                          
construction       220  220  220  220  220  168  176  178  182  196
hvac               100  100  100  100  100  100  100  100  100  100
music_loudspeaker  100  100  100  100  100  100  100  100  100  100
siren               86   91  119  166   71   74   77   80   82   83
traffic_horn       132  142  150  166  205  135  134  118  121  126


In [7]:
# save and verify
out_dir = ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'labels_us8k.csv'

df.to_csv(out_path, index=False)
print('Saved to:', out_path)

# reload and verify
df2 = pd.read_csv(out_path)
print('Reloaded shape  :', df2.shape)
print('Reloaded columns:', df2.columns.tolist())
print(df2.head(5).to_string())

Saved to: c:\Users\LENOVO\Desktop\CityPulse\data\processed\labels_us8k.csv
Reloaded shape  : (6358, 5)
Reloaded columns: ['path', 'source', 'fold', 'original_class', 'app_class']
                                                  path source  fold original_class     app_class
0  data/raw/UrbanSound8K/audio/fold10/100648-1-0-0.wav   us8k    10       car_horn  traffic_horn
1  data/raw/UrbanSound8K/audio/fold10/100648-1-1-0.wav   us8k    10       car_horn  traffic_horn
2  data/raw/UrbanSound8K/audio/fold10/100648-1-2-0.wav   us8k    10       car_horn  traffic_horn
3  data/raw/UrbanSound8K/audio/fold10/100648-1-3-0.wav   us8k    10       car_horn  traffic_horn
4  data/raw/UrbanSound8K/audio/fold10/100648-1-4-0.wav   us8k    10       car_horn  traffic_horn
